In [ ]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.6/496.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 111.2 MB/s eta 0:00:00


In [ ]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 4.5 MB/s eta 0:00:00


In [2]:
import pandas as pd
import re

# 데이터 로드
file_path = "/content/drive/MyDrive/Colab Notebooks/V3_FIN/naver_news_250710.csv"
df = pd.read_csv(file_path)

# 도메인 추출
df['domain'] = df['url'].str.extract(r'https?://(?:www\.)?([^/]+)')

# 전처리 클래스 정의
class NewsCleaner:
    def __init__(self):
        self.cleaning_rules = {
            'yna.co.kr': self.clean_yna,
            'thelec.kr': self.clean_generic,
            'electimes.com': self.clean_electimes,
            'biz.chosun.com': self.clean_chosun,
            'joongang.co.kr': self.clean_joongang,
            'industrynews.co.kr': self.clean_generic,
            'g-enews.com': self.clean_genews,
            'segye.com': self.clean_segye,
            'news.sbs.co.kr': self.clean_sbs,
            'seoulfn.com': self.clean_generic,
            'm.joseilbo.com': self.clean_generic,
            'kyeonggi.com': self.clean_kyeonggi
        }

    def clean(self, text, domain):
        if pd.isnull(text):
            return ""
        cleaner = self.cleaning_rules.get(domain, self.clean_generic)
        return cleaner(text)

    def clean_generic(self, text):
        text = re.sub(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", " ", text)
        text = re.sub(r"[가-힣]{2,4}\s기자", " ", text)
        text = re.sub(r"사진[=:/]?", " ", text)
        text = re.sub(r"이미지\s*확대보기", " ", text)
        text = re.sub(r"\[[^\]]*\]", " ", text)
        text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def clean_yna(self, text):
        text = re.sub(r"[가-힣]{2,4}\s기자", " ", text)
        text = re.sub(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", " ", text)
        text = re.sub(r"\([^)]*연합뉴스[^)]*\)", " ", text)
        text = re.sub(r"\[[^\]]*\]", " ", text)
        text = re.sub(r"제보는\s*카카오톡\s*okjebo.*", " ", text)
        text = re.sub(r"저작권자\s*c\s*연합뉴스.*", " ", text)
        text = re.sub(r"\d{4}[년\.\-\s]?\d{1,2}[월\.\-\s]?\d{1,2}[일\s]?\d{1,2}[시:\s]?\d{1,2}(분)?\s*송고", " ", text)
        return self.clean_generic(text)

    def clean_chosun(self, text):
        text = re.sub(r"[\"“”‘’…·]", " ", text)
        return self.clean_generic(text)

    def clean_joongang(self, text):
        ad_keywords = "중앙광고대상|OOH|중앙선데이|코리아중앙데일리|AD FESTIVAL|중앙일보 수상|광고상"
        if re.search(ad_keywords, text, re.IGNORECASE):
            return ""
        return self.clean_generic(text)

    def clean_genews(self, text):
        text = re.sub(r"\b[가-힣]{2,4}\b\s*$", " ", text)
        return self.clean_generic(text)

    def clean_segye(self, text):
        text = re.sub(r"\(세계비즈\)", " ", text)
        text = re.sub(r"\b[가-힣]{2,4}\b\s*$", " ", text)
        return self.clean_generic(text)

    def clean_sbs(self, text):
        text = re.sub(r"안녕하세요\s*SBS\s*입니다", " ", text)
        text = re.sub(r"잠시 후 SBS.*?전해 드립니다", " ", text)
        text = re.sub(r"(영상취재|영상편집|디자인)\s[가-힣]{2,4}", " ", text)
        text = re.sub(r"SBS\s*(연예뉴스|뉴스)", " ", text)
        return self.clean_generic(text)

    def clean_electimes(self, text):
        text = re.sub(r"\b[가-힣]{2,4}\b(?=\s*이미지|\s*$)", " ", text)
        return self.clean_generic(text)

    def clean_kyeonggi(self, text):
        text = re.sub(r"\(.*?(한양경제|편집자주).*?\)", " ", text)
        text = re.sub(r"\d{4}년\d{2}월\d{2}일\s*\d{2}시\d{2}분\s*송고", " ", text)
        text = re.sub(r"(제보는.*|저작권자.*|무단 전재 재배포.*|AI 학습 및 활용 금지.*)", " ", text)
        return self.clean_generic(text)


    def clean_medicaltoday_content(text):
        text = re.sub(r'^▲.*\n?', '', text, flags=re.MULTILINE)
        text = re.sub(r'\[.*?=.*?기자\]', '', text)
        text = re.sub(r'\([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+\)', '', text)
        text = re.sub(r'저작권자.*?무단전재-재배포 금지', '', text)
        text = re.sub(r'메디컬투데이\s?[가-힣]+\s?\(.*?\)', '', text)
        text = re.sub(r'#\S+', '', text)
        text = re.sub(r'\n+', '\n', text)
        text = re.sub(r'\s{2,}', ' ', text)
        return text.strip()


    def clean_pharmnews(text):
      text = re.sub(r'\[팜뉴스=.+?기자\]', '', text)
      text = re.sub(r'[∙·]', ' ', text)
      text = re.sub(r'\s{2,}', ' ', text)
      text = re.sub(r'\n+', '\n', text)
      return text.strip()


    def clean_thebionews(text):
      text = re.sub(r'^출처\s*:\s*.*?\n?', '', text, flags=re.MULTILINE)
      text = re.sub(r'\[더바이오\s?.+?기자\]', '', text)
      text = re.sub(r'\s{2,}', ' ', text)
      text = re.sub(r'\n+', '\n', text)
      return text.strip()

    def clean_metal_news_text(text):

      text = re.sub(r'(좋아요|도움 돼요|아쉬워요|후속기사 원해요)\s*\d+', '', text)

      text = re.sub(r'\s*사진[=:\s]*[^ \n]+', '', text)
      text = re.sub(r'(편집자주|사진제공\s*[:=]?.*|제공[:=]?.*포스코.*|=사진제공.*|=사진.*)', '', text)
      text = re.sub(r'<[^>]+>', '', text)  # html 태그
      text = re.sub(r'\[[^\]]*\]', '', text)  # 괄호 안 설명
      text = re.sub(r'\([^)]*\)', '', text)  # 괄호 안 내용
      text = re.sub(r'[◆■▲▶▷▶★※→]', '', text)  # 특수문자 패턴 제거
      text = re.sub(r'(■\s*[^.\n]+|포항제철소는\s+|철강업계가\s+)', '', text)
      text = re.sub(r'\n+', '\n', text)
      text = re.sub(r'\s{2,}', ' ', text)
      return text.strip()


# 전처리 적용
cleaner = NewsCleaner()
df['content'] = df.apply(lambda row: cleaner.clean(row['content'], row['domain']), axis=1)

# 저장
output_path = "/content/drive/MyDrive/Colab Notebooks/V3_FIN/naver_news_250710_cleaned.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
output_path


'/content/drive/MyDrive/Colab Notebooks/V3_FIN/naver_news_250710_cleaned.csv'

In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
from konlpy.tag import Okt

# 파일 로딩
file_path = "/content/drive/MyDrive/Colab Notebooks/v3_news_june_cut_cleaned.csv"
news_df = pd.read_csv(file_path)
lexicon_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/finance_sentiment_cleaned_final_700.csv", encoding='utf-8')
lexicon_df['score'] = pd.to_numeric(lexicon_df['score'], errors='coerce')
lexicon_df.dropna(subset=['score'], inplace=True)
lexicon = dict(zip(lexicon_df['word'], lexicon_df['score']))

# 산업군별 특화 감정단어 로딩
industry_word_df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/word_of_industry_v3.xlsx")
industry_word_dict = industry_word_df.groupby('industry')['word'].apply(set).to_dict()
# 가중치 비율 1.5
industry_weight = 1.5

# 전처리 요소
okt = Okt()
NEGATION_WORDS = ['않다', '못하다', '없다', '아니다', '줄이다']
NEGATIVE_CONTEXT = ['문제', '위기', '불안', '침체', '파산', '위협', '적자', '부진', '하락', '리스크']
POSITIVE_RESOLVE = ['해결', '극복', '해소', '회복', '정상화', '반등', '개선']
NEGATE_IGNORE_WORDS = ['전쟁', '위기', '불확실', '파산', '침체', '리스크']

def split_sentences(text):
    return re.split(r'(?<=[.!?])\s+', str(text))

def compute_sentiment_score(text, lexicon, industry=None):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, ''

    text_len = len(text)
    if text_len <= 625:
        per_word_cap = 1.5
    elif text_len <= 1360:
        per_word_cap = 2.5
    else:
        per_word_cap = 3.5

    word_score_dict = defaultdict(float)
    word_count_dict = defaultdict(int)

    industry_words = industry_word_dict.get(industry, set())

    for phrase, score in lexicon.items():
        if ' ' in phrase and phrase in text:
            base_score = score
            if phrase in industry_words:
                base_score *= industry_weight

            capped = max(min(base_score, per_word_cap), -per_word_cap)
            word_score_dict[phrase] += capped
            word_score_dict[phrase] = max(min(word_score_dict[phrase], per_word_cap), -per_word_cap)
            word_count_dict[phrase] += 1
            text = text.replace(phrase, '')

    for sentence in split_sentences(text):
        try:
            tokens = okt.pos(sentence, stem=True)
        except:
            continue

        words = [w for w, _ in tokens]
        word_freq = Counter(words)

        for i, (word, pos) in enumerate(tokens):
            if pos not in ['Noun', 'Adjective', 'Verb']:
                continue
            if word not in lexicon:
                continue

            base_score = lexicon[word]
            if word in industry_words:
                base_score *= industry_weight

            count = word_freq[word]

            if word in NEGATE_IGNORE_WORDS:
                adjusted_score = base_score
            else:
                if pos in ['Verb', 'Adjective']:
                    negate = any(
                        (i + j < len(tokens) and tokens[i + j][0] in NEGATION_WORDS) or
                        (i - j >= 0 and tokens[i - j][0] in NEGATION_WORDS)
                        for j in range(1, 2)
                    )
                else:
                    negate = False
                adjusted_score = base_score * (-1 if negate else 1)

            log_score = adjusted_score * np.log10(1 + count) * 0.8
            capped_log_score = max(min(log_score, per_word_cap), -per_word_cap)

            word_score_dict[word] += capped_log_score
            word_score_dict[word] = max(min(word_score_dict[word], per_word_cap), -per_word_cap)
            word_count_dict[word] += count

        if any(w in words for w in NEGATIVE_CONTEXT) and any(p in words for p in POSITIVE_RESOLVE):
            word_score_dict['보정'] += 1.5
            word_score_dict['보정'] = min(word_score_dict['보정'], per_word_cap)
            word_count_dict['보정'] += 1

    total_score = sum(word_score_dict.values())
    hit_summary = sorted(
        [f"{word}:{round(word_score_dict[word], 3)}({word_count_dict[word]})" for word in word_score_dict],
        key=lambda x: abs(float(x.split(":")[1].split("(")[0])),
        reverse=True
    )

    return round(total_score, 3), ', '.join(hit_summary)

# 감정 점수 계산 적용 (산업군 기반 가중치 포함)
news_df[['sentiment_score', 'hit_words']] = news_df.apply(
    lambda row: pd.Series(compute_sentiment_score(row['content'], lexicon, row.get('industry'))),
    axis=1
)

# 정규화 점수 계산
news_df['content_length'] = news_df['content'].str.len().replace(0, np.nan)
news_df['normalized_score'] = news_df['sentiment_score'] / np.log10(news_df['content_length'] + 1)

# 결과 저장
news_df.to_csv("/content/drive/MyDrive/Colab Notebooks/v3_news_june_cut_cleaned_senti.csv", index=False, encoding="utf-8-sig")